[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_demo04_walmart.ipynb)

# MATH840 — Week 4 practice: Walmart weekly sales

45 Walmart stores, weekly sales from February 2010 to October 2012, with the covariates the retailer
actually had: a holiday flag, temperature, fuel price, CPI and unemployment.

We forecast this series through its decomposition — and then ask whether the decomposition deserved
to be believed. Three things make this series worth the hour:

| Block | What happens, and what it teaches |
|---|---|
| 1 | The holiday flag that ships with the data misses the two biggest weeks in the whole series |
| 2 | The recipe works: decomposition beats every simple method — but not by much, and the ordering is the lesson |
| 3 | STL reports $F_S = 0.99$ on a series three years long. The number is real; what it measures is not seasonality |

**This notebook is not marked and not part of Lab 4.** Your own series has fifteen years or more of
history, so the trap in block 3 cannot bite you there — but the judgement is the reason you will not
fall for it later.

## 0. Setup

In [ ]:
!pip install -q statsforecast utilsforecast coreforecast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

walmart = pd.read_csv(f"{BASE}/walmart.csv", parse_dates=["ds"])
print(f"{walmart['unique_id'].nunique()} stores x {len(walmart) // walmart['unique_id'].nunique()} "
      f"weeks, {walmart['ds'].min().date()} to {walmart['ds'].max().date()}")
walmart.head(3)

We forecast the **chain**: all 45 stores added up, in millions per week. Aggregating is a decision,
not a formality — a single store is noisier, and the noise is what a model has to avoid learning.

In [ ]:
total = (walmart.groupby("ds", as_index=False)
         .agg(y=("y", "sum"), holiday=("holiday", "max"), temperature=("temperature", "mean")))
total["y"] = total["y"] / 1e6            # millions of dollars per week
y = total["y"].to_numpy(float)
dates = total["ds"]
M, H = 52, 13                            # weekly season; a quarter of a year held out

store_one = walmart.query("unique_id == 1").set_index("ds")["y"] / 1e6

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(dates, y, color="#314f4f", linewidth=1)
axes[0].set_title("All 45 stores, weekly sales")
axes[0].set_ylabel("$m")
axes[1].plot(store_one.index, store_one.to_numpy(), color="#6A5ACD", linewidth=1)
axes[1].set_title("Store 1 alone")
axes[1].set_ylabel("$m")
plt.tight_layout()
plt.show()

print(f"{len(y)} weeks = {len(y) / M:.2f} cycles of {M}")
print(f"chain: mean {y.mean():.1f}, min {y.min():.1f}, max {y.max():.1f}")
print(f"relative week-to-week noise: chain {np.std(np.diff(y)) / y.mean():.3f}, "
      f"store 1 {np.std(np.diff(store_one.to_numpy())) / store_one.mean():.3f}")

## 1. The holiday flag, before we model anything

The file comes with `holiday`, marking the weeks of Super Bowl, Labour Day, Thanksgiving and
Christmas. Plot it against the sales it is supposed to explain.

In [ ]:
fig, ax = plt.subplots()
ax.plot(dates, y, color="#314f4f", linewidth=1, label="weekly sales")
flagged = total.query("holiday == 1")
ax.scatter(flagged["ds"], flagged["y"], color="#e64173", zorder=3, s=45, label="holiday = 1")
top = total.nlargest(4, "y")
ax.scatter(top["ds"], top["y"], facecolors="none", edgecolors="#20B2AA", s=160, linewidth=2,
           label="four biggest weeks")
ax.legend()
ax.set_title("What the flag marks, and where the money actually is")
ax.set_ylabel("$m")
plt.show()

print(total.nlargest(6, "y").assign(week=lambda d: d.ds.dt.strftime("%Y-%m-%d"))
      [["week", "y", "holiday"]].to_string(index=False))
print(f"\nflagged weeks: {int(total.holiday.sum())} of {len(total)}")
print(f"mean sales, flagged {flagged.y.mean():.1f} vs {total.query('holiday == 0').y.mean():.1f} "
      f"elsewhere ({flagged.y.mean() / total.query('holiday == 0').y.mean() - 1:+.1%})")
print(f"correlation of the flag with sales: {np.corrcoef(y, total.holiday)[0, 1]:+.2f}")

::: Discussion
**The two biggest weeks of the entire series are not flagged as holidays. Why not — and is the flag
wrong?**

Then the practical question: if you were handed this flag as a regressor for a Christmas forecast,
what would you do with it?
:::

In [ ]:
# The weeks before 25 December, which is when the shopping happens.
total["prechristmas"] = (total.ds.dt.month.eq(12) & total.ds.dt.day.between(10, 26)).astype(int)

print("the pre-Christmas weeks:")
print(total.query("prechristmas == 1").assign(week=lambda d: d.ds.dt.strftime("%Y-%m-%d"))
      [["week", "y", "holiday"]].to_string(index=False))
print(f"\nlift over an average week: {total.query('prechristmas == 1').y.mean() / y.mean() - 1:+.0%}")
print(f"correlation with sales:  dataset flag {np.corrcoef(y, total.holiday)[0, 1]:+.2f}   "
      f"pre-Christmas indicator {np.corrcoef(y, total.prechristmas)[0, 1]:+.2f}")
print(f"temperature: {np.corrcoef(y, total.temperature)[0, 1]:+.2f}")

The flag is not wrong — it marks holidays. It is **mistimed** for the effect: Thanksgiving week is
flagged, the two weeks of Christmas shopping are not, and one date in the flag is the week *after*
Christmas, when sales have already collapsed.

A covariate has to line up with the mechanism, not with the calendar's own labels. That is Week 7's
subject, and this is the cheapest possible illustration of it.

## 2. The recipe: decomposition against the simple methods

Split off the last 13 weeks, run the four simple methods and the decomposition forecast, score
everything with MASE on the same scale. Nothing here looks at the validation window until it is
scored.

In [ ]:
def benchmark_forecasts(history, h, m):
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))
mase = lambda actual, fc: float(np.mean(np.abs(np.asarray(actual) - np.asarray(fc))) / SCALE)
rmse = lambda actual, fc: float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(fc)) ** 2)))


def decomposition_forecast(history, h, m, level_method, seasonal=13, robust=True):
    fit = STL(history, period=m, seasonal=seasonal, robust=robust).fit()
    adjusted = history - fit.seasonal
    season = np.array([fit.seasonal[-m + (i % m)] for i in range(h)])
    return benchmark_forecasts(adjusted, h, m)[level_method] + season, fit


train, valid = y[:-H], y[-H:]
print(f"train {len(train)} weeks ({len(train) / M:.2f} cycles), "
      f"valid {H} weeks: {dates.iloc[-H].date()} to {dates.iloc[-1].date()}")

scores = {name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
          for name, fc in benchmark_forecasts(train, H, M).items()}
for level_method in ("naive", "drift", "mean"):
    fc, _ = decomposition_forecast(train, H, M, level_method)
    scores[f"STL + {level_method}"] = {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}

table = pd.DataFrame(scores).T.sort_values("mase")
print()
print(table.round(3).to_string())

In [ ]:
best = table.index[0]
best_fc = (decomposition_forecast(train, H, M, best.split(" + ")[1])[0] if best.startswith("STL")
           else benchmark_forecasts(train, H, M)[best])
snaive_fc = benchmark_forecasts(train, H, M)["snaive"]

fig, ax = plt.subplots()
ax.plot(dates.iloc[-60:], y[-60:], color="#314f4f", linewidth=1, label="actual")
ax.plot(dates.iloc[-H:], best_fc, color="#20B2AA", linewidth=2, label=best)
ax.plot(dates.iloc[-H:], snaive_fc, "--", color="#e64173", linewidth=2, label="snaive")
ax.legend()
ax.set_title("The last 60 weeks, and two forecasts of the last 13")
ax.set_ylabel("$m")
plt.show()

::: Discussion
**The decomposition wins, `snaive` is right behind it, and plain `naive` is three times worse. What
does that ordering tell you about this series?**

And: `snaive` here copies the same week from **one** year ago, because that is all there is. How much
evidence is that?
:::

## 3. $F_S = 0.99$, and why you should not believe it

Now decompose the whole series and compute the strength of seasonality the way Lab 3 taught.

In [ ]:
fit = STL(y, period=M, seasonal=13, robust=True).fit()
S, T_, R = fit.seasonal, fit.trend, fit.resid
F_S = max(0.0, 1 - R.var() / (S + R).var())
F_T = max(0.0, 1 - R.var() / (T_ + R).var())
print(f"F_S = {F_S:.2f}   F_T = {F_T:.2f}")

fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
for ax, (label, comp) in zip(axes, [("sales", y), ("trend", T_), ("seasonal", S), ("remainder", R)]):
    ax.plot(dates, comp, linewidth=0.9)
    ax.set_ylabel(label)
fig.suptitle("STL(period=52) on 143 weeks")
plt.tight_layout()
plt.show()

$F_S = 0.99$ would normally mean *this series is almost entirely seasonal*. Before believing it, count
the evidence behind each seasonal value.

In [ ]:
weeks = dates.dt.isocalendar().week.to_numpy()
per_week = pd.DataFrame({"week": weeks, "seasonal": S})
counts = per_week.groupby("week").size()
spread = per_week.groupby("week")["seasonal"].std()

print(f"observations behind each week-of-year: min {counts.min()}, median {counts.median():.0f}, "
      f"max {counts.max()}")
print(f"seasonal component: range {S.max() - S.min():.1f}")
print(f"spread of the seasonal values *within* a week-of-year: median sd {spread.median():.3f} "
      f"= {spread.median() / (S.max() - S.min()):.1%} of that range")
print("\nIn other words: each seasonal value is fitted to two or three observations, and it passes")
print("through them almost exactly. That is not an estimate of a season - it is a copy of the data.")

::: Discussion
**So is the seasonality real?**

Two answers are defensible, and the difference between them matters:

- December really does lift sales — that is a mechanism, not an artefact;
- but $F_S = 0.99$ measured on 2.75 cycles is mostly memorisation, and the same number on your own
  fifteen-year series would mean something completely different.

**What would you need to separate the two?** And what does this say about reporting $F_S$ without
saying how long the series is?
:::

::: {.callout}
**The rule worth writing down:** a seasonal period of $m$ wants at least three complete cycles before
its seasonal component is worth reading, and more before it is worth trusting. 143 weeks against
$m = 52$ is 2.75 — which is why every series in the Track B pool has fifteen years or more.
:::

## 4. What to take home

1. Decomposition earns its place here, but the margin is thin: 6% better than `snaive` in MASE, while
   `naive` is three times worse than either. The gap between the two seasonal methods is small because
   both are reading the same two Decembers.
2. A high $F_S$ is a claim about a ratio of variances, not a certificate. Ask how many cycles it was
   computed from before you quote it.
3. Covariates have to line up with the mechanism. The holiday flag in this file correlates +0.17 with
   sales; an indicator built for the weeks when people actually shop correlates +0.70. Week 7 is
   about doing that properly.
4. Aggregation is a modelling decision: the chain is far smoother than any single store, and a
   smoother series is easier to forecast — but it answers a different question.